In [296]:
!pip install iterative-stratification tqdm scikit-learn numpy torch transformers pandas scikit-multilear datetime

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not find a version that satisfies the requirement scikit-multilear (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for scikit-multilear


In [297]:
import datetime
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import csv

start_time = datetime.datetime.now()
print(f"Script started at {start_time}")

timing_info = {}
timing_info['Start timing'] = start_time

Script started at 2025-12-03 10:40:53.992047


In [298]:
# Configuration parameters
two_emotions = False
file_name = "GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 5
emotions = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

if two_emotions:
    file_name = "GroundTruthJoySadness.csv"
    emotions = ["Joy", "Sadness"]

# Undersampling configuration
undersample = False
cut_off = 150

# Class weight configuration
class_weight = True
function = 2  # options: BCEWithLogitsLoss (0), FocalLoss (1) or CrossEntropyLoss (2)
reduction = "sum" # options: mean or sum
gamma = 2.0  # for Focal Loss, when incrementing the value, focus more on hard examples.

# SMOTE configuration
smote = False

In [299]:
# Configure output file path
suffixes = []
if undersample:
    suffixes.append("undersample")
if class_weight:
    suffixes.append("class_weight")
    if function == 0:
        suffixes.append("bce_with_logits_loss")
    elif function == 1:
        suffixes.append("focal_loss")
    elif function == 2:
        suffixes.append("cross_entropy_loss")

    suffixes.append(f"reduction_{reduction}")
if smote:
    suffixes.append("smote")

suffix_str = "_" + "_".join(suffixes) if suffixes else ""
output_path_csv = f"results/results_{checkpoint}_with_folds_{num_folds}__{suffix_str}.csv"
if two_emotions:
    output_path_csv = f"results/results_2e_{checkpoint}_with_folds_{num_folds}__{suffix_str}.csv"

In [300]:
df = pd.read_csv(file_name, quotechar='"', sep=';')
#df = df[df['Reject'] != 1]  # Filter out rows where Reject is 1

In [301]:
if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df = df.reset_index(drop=True)
    pos_counts = df[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df.index[df[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

In [302]:
X = df[['sentence']].values
y = df[emotions].values

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [303]:
pos_weight = None

if class_weight:
    start_time_class_weight = datetime.datetime.now()
    num_pos = df[emotions].sum().values
    num_neg = len(df) - num_pos

    weights = num_neg / (num_pos + 1e-6)
    pos_weight = torch.tensor(weights, dtype=torch.float).to(device)

    print("Label counts (Positives):\n", num_pos)
    print("Computed pos_weights (Neg/Pos):\n", pos_weight)

    print("Weights per emotion:")
    for emotion, w in zip(emotions, pos_weight.tolist()):
        print(f"{emotion:12s}: {w:.3f}")

    timing_info['Class weight calculation time'] = datetime.datetime.now() - start_time_class_weight

Label counts (Positives):
 [372 152  24  54 303  50  27 182  88]
Computed pos_weights (Neg/Pos):
 tensor([ 1.9892,  6.3158, 45.3333, 19.5926,  2.6700, 21.2400, 40.1852,  5.1099,
        11.6364])
Weights per emotion:
Joy         : 1.989
Trust       : 6.316
Fear        : 45.333
Surprise    : 19.593
Sadness     : 2.670
Disgust     : 21.240
Anger       : 40.185
Anticipation: 5.110
Neutral     : 11.636


In [304]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiLabelFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, reduction='mean'):
        super(MultiLabelFocalLoss, self).__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction=reduction,
            pos_weight=self.pos_weight
        )

        pt = torch.exp(-bce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [305]:
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

In [306]:
def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return batch_inputs

In [307]:
def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(emotions),
        problem_type="multi_label_classification"
    )
    model.to(device)
    return model

In [308]:
multilabel_stratified_in_folds = MultilabelStratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
all_metrics = []
fold_results = []

In [309]:
for fold, (train_idx, val_idx) in enumerate(multilabel_stratified_in_folds.split(X, y)):
    timing_fold = datetime.datetime.now()
    print(f"\n===== Fold {fold + 1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, emotions)
    val_dataset = EmotionDataset(val_df, tokenizer, emotions)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    model = create_model()

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))

    if class_weight and function == 0:
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 1:
        loss_fct = MultiLabelFocalLoss(gamma=2.0, pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 2:
        loss_fct = torch.nn.CrossEntropyLoss(weight=pos_weight, reduction=reduction)

    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            if class_weight:
                logits = outputs.logits
                loss = loss_fct(logits, batch["labels"].float())
            else:
                loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = (torch.sigmoid(logits).numpy() > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    sentences = val_df["sentence"].values


    def binvec_to_names(vec):
        names = [emotions[i] for i, v in enumerate(vec) if int(v) == 1]
        return ", ".join(names) if names else "None"


    true_emotions = [binvec_to_names(row) for row in y_true]
    pred_emotions = [binvec_to_names(row) for row in y_pred]

    df_predictions = pd.DataFrame({
        "sentence": sentences,
        "true_emotions": true_emotions,
        "pred_emotions": pred_emotions
    })

    # show a quick preview and optionally save per fold
    #print(df_predictions.head(20).to_string(index=False))
    df_predictions.to_csv(f"predictions/predictions_fold_{fold + 1}_out_of_{num_folds}.csv", index=False)

    fold_results.append({
        "fold": fold + 1,
        "y_true": y_true,
        "y_pred": y_pred
    })

    metrics = {
        "Subset Accuracy (Exact Match)": accuracy_score(y_true, y_pred),
        "Micro Precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro Recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro F1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

    # --- Print the Summary ---
    print(f"\n=== Metrics for fold {fold + 1} ===")
    for k, v in metrics.items():
        print(f"{k:25s}: {v:.4f}")

    all_metrics.append(metrics)
    end_timing_fold = datetime.datetime.now()
    timing_info[f'Fold {fold + 1} time'] = end_timing_fold - timing_fold


===== Fold 1 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 336/336 [06:18<00:00,  1.13s/it]

100%|██████████| 336/336 [02:59<00:00,  2.06it/s]Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Metrics for fold 1 ===
Subset Accuracy (Exact Match): 0.0000
Micro Precision          : 0.2306
Micro Recall             : 0.8835
Micro F1                 : 0.3658
Macro Precision          : 0.2246
Macro Recall             : 0.8877
Macro F1                 : 0.3187

===== Fold 2 =====


100%|██████████| 336/336 [03:09<00:00,  2.18it/s]Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Metrics for fold 2 ===
Subset Accuracy (Exact Match): 0.0135
Micro Precision          : 0.2510
Micro Recall             : 0.7849
Micro F1                 : 0.3803
Macro Precision          : 0.2623
Macro Recall             : 0.8411
Macro F1                 : 0.3386

===== Fold 3 =====


...


=== Metrics for fold 3 ===
Subset Accuracy (Exact Match): 0.0404
Micro Precision          : 0.2552
Micro Recall             : 0.8400
Micro F1                 : 0.3914
Macro Precision          : 0.2355
Macro Recall             : 0.8209
Macro F1                 : 0.3327

===== Fold 4 =====


100%|██████████| 336/336 [03:50<00:00,  2.11it/s]


=== Metrics for fold 4 ===
Subset Accuracy (Exact Match): 0.0000
Micro Precision          : 0.2403
Micro Recall             : 0.8538
Micro F1                 : 0.3750
Macro Precision          : 0.2340
Macro Recall             : 0.8462
Macro F1                 : 0.3244

===== Fold 5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 336/336 [03:53<00:00,  1.44it/s]

100%|██████████| 336/336 [03:19<00:00,  2.06it/s]


=== Metrics for fold 5 ===
Subset Accuracy (Exact Match): 0.0045
Micro Precision          : 0.2625
Micro Recall             : 0.9036
Micro F1                 : 0.4069
Macro Precision          : 0.2290
Macro Recall             : 0.8246
Macro F1                 : 0.3327


In [310]:
print("\n===== Summary per fold =====")
timing_results = datetime.datetime.now()

df_metrics = pd.DataFrame(all_metrics).T  # transpose to have metrics as rows
df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
print("\n=== Overall Metrics per Fold ===")
print(df_metrics.to_markdown(floatfmt=".4f"))


===== Summary per fold =====

=== Overall Metrics per Fold ===
|                               |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:------------------------------|---------:|---------:|---------:|---------:|---------:|
| Subset Accuracy (Exact Match) |   0.0000 |   0.0135 |   0.0404 |   0.0000 |   0.0045 |
| Micro Precision               |   0.2306 |   0.2510 |   0.2552 |   0.2403 |   0.2625 |
| Micro Recall                  |   0.8835 |   0.7849 |   0.8400 |   0.8538 |   0.9036 |
| Micro F1                      |   0.3658 |   0.3803 |   0.3914 |   0.3750 |   0.4069 |
| Macro Precision               |   0.2246 |   0.2623 |   0.2355 |   0.2340 |   0.2290 |
| Macro Recall                  |   0.8877 |   0.8411 |   0.8209 |   0.8462 |   0.8246 |
| Macro F1                      |   0.3187 |   0.3386 |   0.3327 |   0.3244 |   0.3327 |


In [311]:
emotion_metrics = {emotion: {"Precision": [], "Recall": [], "F1": [], "Accuracy": []} for emotion in emotions}

for fold_data in fold_results:
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    acc = [accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]

    for i, emotion in enumerate(emotions):
        emotion_metrics[emotion]["Precision"].append(prec[i])
        emotion_metrics[emotion]["Recall"].append(rec[i])
        emotion_metrics[emotion]["F1"].append(f1[i])
        emotion_metrics[emotion]["Accuracy"].append(acc[i])

In [312]:
print("\n===== Per Emotion Metrics per Fold =====")
for emotion, metrics_dict in emotion_metrics.items():
    df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
    print(f"\n--- {emotion} ---")
    print(df_emotion.to_markdown(floatfmt=".4f"))


===== Per Emotion Metrics per Fold =====

--- Joy ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.5276 |   0.6421 |   0.5172 |   0.5798 |   0.5106 |
| Recall    |   0.9054 |   0.8243 |   0.8108 |   0.9200 |   0.9600 |
| F1        |   0.6667 |   0.7219 |   0.6316 |   0.7113 |   0.6667 |
| Accuracy  |   0.6982 |   0.7883 |   0.6861 |   0.7489 |   0.6757 |

--- Trust ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.2000 |   0.2117 |   0.2889 |   0.2320 |   0.2596 |
| Recall    |   0.9677 |   0.9355 |   0.8667 |   0.9667 |   0.9000 |
| F1        |   0.3315 |   0.3452 |   0.4333 |   0.3742 |   0.4030 |
| Accuracy  |   0.4550 |   0.5045 |   0.6951 |   0.5650 |   0.6396 |

--- Fear ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|------

In [313]:
print("\n===== Average Per-Emotion Metrics =====")
df_emotion_avg = pd.DataFrame({
    emotion: {metric: np.mean(values) for metric, values in metrics.items()}
    for emotion, metrics in emotion_metrics.items()
})
print(df_emotion_avg.T.to_markdown(floatfmt=".4f"))


===== Average Per-Emotion Metrics =====
|              |   Precision |   Recall |     F1 |   Accuracy |
|:-------------|------------:|---------:|-------:|-----------:|
| Joy          |      0.5555 |   0.8841 | 0.6796 |     0.7194 |
| Trust        |      0.2384 |   0.9273 | 0.3774 |     0.5718 |
| Fear         |      0.0546 |   0.8800 | 0.1027 |     0.6709 |
| Surprise     |      0.1025 |   0.7945 | 0.1815 |     0.6511 |
| Sadness      |      0.5901 |   0.7563 | 0.6545 |     0.7869 |
| Disgust      |      0.0893 |   0.8600 | 0.1616 |     0.5971 |
| Anger        |      0.0690 |   0.7333 | 0.1258 |     0.7455 |
| Anticipation |      0.3006 |   0.9290 | 0.4520 |     0.6206 |
| Neutral      |      0.1338 |   0.8320 | 0.2296 |     0.5485 |


In [314]:
df_metrics = pd.DataFrame(all_metrics)
print("\n===== Cross-validation mean results =====")
print(df_metrics.mean())

timing_info[f'Printing results'] = datetime.datetime.now() - timing_results

with open(output_path_csv, mode="w", newline="") as f:
    writer = csv.writer(f)

    # ===== SUMMARY PER FOLD =====
    writer.writerow(["===== Summary per fold ====="])
    df_metrics = pd.DataFrame(all_metrics).T  # metrics as rows
    df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
    df_metrics.reset_index(inplace=True)
    df_metrics.rename(columns={"index": "Metric"}, inplace=True)
    df_metrics.to_csv(f, index=False, float_format="%.4f")
    writer.writerow([])  # blank line

    # ===== PER-EMOTION METRICS PER FOLD =====
    writer.writerow(["===== Per Emotion Metrics per Fold ====="])
    for emotion, metrics_dict in emotion_metrics.items():
        writer.writerow([f"--- {emotion} ---"])
        df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
        df_emotion.reset_index(inplace=True)
        df_emotion.rename(columns={"index": "Metric"}, inplace=True)
        df_emotion.to_csv(f, index=False, float_format="%.4f")
        writer.writerow([])  # blank line

    # ===== AVERAGE PER EMOTION =====
    writer.writerow(["===== Average Per-Emotion Metrics ====="])
    df_emotion_avg = pd.DataFrame({
        emotion: {metric: np.mean(values) for metric, values in metrics.items()}
        for emotion, metrics in emotion_metrics.items()
    })
    df_emotion_avg = df_emotion_avg.T.reset_index().rename(columns={"index": "Emotion"})
    df_emotion_avg.to_csv(f, index=False, float_format="%.4f")
    writer.writerow([])

    # ===== CROSS-VALIDATION MEAN RESULTS =====
    writer.writerow(["===== Cross-validation Mean Results ====="])
    df_metrics = pd.DataFrame(all_metrics)
    df_mean = df_metrics.mean().to_frame(name="Mean").reset_index().rename(columns={"index": "Metric"})
    df_mean.to_csv(f, index=False, float_format="%.4f")

    # ===== CLASS WEIGHTS =====
    if class_weight:
        writer.writerow([])
        writer.writerow(["===== Class Weights per Emotion ====="])
        writer.writerow(["Emotion", "Weight"])
        for emotion, weight in zip(emotions, pos_weight.tolist()):
            writer.writerow([emotion, f"{weight:.6f}"])
        writer.writerow([])

    writer.writerow(["===== Timings ====="])
    for k, v in timing_info.items():
        writer.writerow([k, str(v)])

    writer.writerow(["Total duration", str(datetime.datetime.now() - start_time)])

print(f"\n✅ All tables saved in one CSV file: {os.path.abspath(output_path_csv)}")


===== Cross-validation mean results =====
Subset Accuracy (Exact Match)    0.011675
Micro Precision                  0.247908
Micro Recall                     0.853153
Micro F1                         0.383872
Macro Precision                  0.237085
Macro Recall                     0.844071
Macro F1                         0.329418
dtype: float64

✅ All tables saved in one CSV file: /Users/carlotacatot/Docs/pHd/EmotionClassification/results/results_bert-base-cased_with_folds_5___class_weight_cross_entropy_loss_reduction_sum.csv
